# Discovery — checkpointed, batch-by-batch

Run the model-discovery task queue **one batch at a time**, reviewing between batches. Each batch
evaluates a set of *testing conditions* (architecture × feature selector × CV track) under
forward-chained cross-validation and reports pooled RMSEP with a bootstrap 95% CI. Configure which
**data** and which **conditions** to run in sections 1–2, then run the batch cells in order.

---

**Where this fits — the ElectroPyCal analysis pipeline** (`pip install electropycal`):

1. `raw_spectra_review` — inspect raw EIS/FSCV spectra; choose the analysis band & QC stringency
2. `quality_filtering_dashboard` — set QC gates, see which `(channel, timepoint)` sensors pass, save the shared config
3. `stabilization_review` — confirm each channel's FSCV response stabilized (optional / parallel)
4. `diagnostics_review` — characterize the featureset before modeling — *is recalibration well-posed?*
5. ➡️ **`discovery_checkpointed` — run the model-discovery task queue batch-by-batch under nested CV  ← you are here**
6. `discovery_results_review` — inspect a finished discovery run; pick a model
7. `deployment_domain_shift` — freeze the chosen model; recalibrate in-vivo data; monitor CORAL domain shift

Consistency knobs (band, QC gates) are shared across notebooks via `<ROOT>/electropycal_analysis_config.json`. See `docs/USAGE.md`.


## Setup — run first

Installs `electropycal` from PyPI. On Colab it also mounts Google Drive, so you can point
`ROOT` at raw data in your own Drive; skip the mount if your data is local.

```
pip install electropycal
```

After installing on Colab, do **Runtime → Restart session** so the new module loads.


In [ ]:
import sys

if "google.colab" in sys.modules:
    !pip install -q electropycal
    from google.colab import drive
    drive.mount("/content/drive")     # only needed if ROOT points into your Drive


## Setup — data source & featureset
Point `ROOT` at a raw PSTrace directory (Google Drive) or leave `None` for a synthetic demo.
`extract_dataset` builds the QC-gated featureset (`acceptance="monotonic"`, `peak_method="direct"`
— recommended for real broad-DA-peak data); **this is the slow cell** (it parses every EIS/FSCV
file once). `PROFILE = FAST` for a quick smoke; use `Profile()` for a real full run. It **extracts once** to `<ROOT>/discovery_featureset__<hash>.parquet` and reloads that on later runs, so only the *first* run pays the raw-parsing cost; the cache key includes the extraction params and the feature-schema version, so it rebuilds automatically when either changes (or set `FORCE_REEXTRACT=True`). This is still standalone — it caches only this notebook's own extract from raw, never another notebook's parquet.

Features are **D0-normalized per sensor** (drift from each sensor's earliest timepoint
— the same transform deployment applies in vivo) and **`NormIpeak < 0` dose rows are dropped**
(`drop_negative`, non-physical) before the featureset. Both are printed below.

**Shared config:** `BAND` and the QC / extraction parameters come from
`<ROOT>/electropycal_analysis_config.json` via `load_analysis_config(ROOT)` — the *same* values you
set once in `raw_spectra_review` (its "Save shared config" cell) and preview in
`quality_filtering_dashboard`, so the three notebooks can't drift apart. Falls back to defaults
(`band="auto"`) if none has been saved. Override any field inline in the setup cell for a one-off.

Two footguns are handled automatically so you don't get silently-empty results:
- **`BAND` is data-driven when set to `"auto"`** — `recommended_band(ROOT)` puts the upper bound just
  below the 10th-percentile inductive onset, so the lowest-onset ~10% of channels fall in-band and
  fail EIS.1 as genuinely poor electrodes rather than shrinking the band for everyone. A fixed 100 kHz
  band would sit above real onsets and drop most channels under the all-in-band EIS.1 check.
- **`min_train_times` auto-caps** to `n_timepoints − 1` in section 1 — forward-chained CV needs at
  least `min_train_times + 1` distinct timepoints, so on a 3-timepoint dataset the default of 3
  would yield **0 folds / nan**. The cap is printed when it fires.

**Standalone by design.** This notebook re-reads the **raw** PSTrace directory and **re-applies the QC gates + feature extraction itself** (the cell below) — it does *not* load any parquet produced by `quality_filtering_dashboard` or `diagnostics_review`, so it runs on its own. The only thing shared with those notebooks is the **parameter** file `electropycal_analysis_config.json` (band + QC thresholds, not data), and it falls back to defaults if that file is absent. (For a large raw store you'd instead extract once to `featureset.parquet` and point discovery at it — see `docs/RUNNING_AT_SCALE.md`.)

In [ ]:
%matplotlib inline
# Pin BLAS/OpenMP to 1 thread BEFORE importing numpy so joblib process-parallelism
# (parallel extraction + discovery fold-parallelism) does not oversubscribe the cores.
import os
for _v in ("OMP_NUM_THREADS","OPENBLAS_NUM_THREADS","MKL_NUM_THREADS",
           "VECLIB_MAXIMUM_THREADS","NUMEXPR_NUM_THREADS"):
    os.environ.setdefault(_v, "1")
import tempfile, time, pathlib, hashlib, json, numpy as np, pandas as pd, matplotlib.pyplot as plt
from electropycal.viz import set_pub_style
from electropycal.data.pstrace import parse_folder
from electropycal.discovery.config import RunData, Profile, FAST, baseline_queue, effective_min_train_times
from electropycal.discovery.runner import run_condition
from electropycal.discovery.scheduler import auto_flag_conditions
from electropycal.overview import (plot_qc_grid, plot_normipeak_per_sensor,
    plot_normipeak_per_device, conditions_table)
from electropycal.discovery.batch import BatchRunner
set_pub_style()

# --- input: a raw PSTrace directory, or None for the synthetic demo ---
ROOT = None   # <- your own PSTrace export directory; None uses the shipped demo
# demo_input() locates demo/ from ANY working directory -- the repo root, notebooks/,
# or anywhere else -- and synthesizes an equivalent tree when no checkout is reachable
# (e.g. Colab). A relative literal here breaks in whichever cwd it was not written for.
from electropycal._demo import demo_input
ROOT = demo_input("in_vitro", ROOT)
print("ROOT =", ROOT)
N_JOBS = 4                     # worker processes for parallel extraction + discovery folds (-1 = all cores; lower if RAM-limited)
REAL_RUN = False               # False = FAST smoke; True = full Profile with fold-parallelism
SEEDS = (0,)                   # set e.g. (0, 1, 2) to seed-average the stochastic selectors (CARS/MI)
PROFILE = Profile(n_jobs=N_JOBS, seeds=SEEDS) if REAL_RUN else FAST   # FAST = smoke; full = the real run
TARGET = "normipeak"           # "normipeak" (per-dose, default) | "sensitivity" (per sensor-timepoint
                               #   dose-response slope — the recalibration signal; see §1)

def DLAB(tp): return f"D{int(tp)}"

# device -> devicetype (neurostring / cfme / ...) from the session folder names (fast; real data only)
DTYPE_OF = {}
if ROOT is not None:
    for _f in sorted(p for p in pathlib.Path(ROOT).iterdir() if p.is_dir()):
        _m = parse_folder(_f.name)
        if _m and _m.get("testtype") == "signal":
            for _csv in _f.glob("*_fscv_0nM.csv"):
                _dev = _csv.name.split("_", 1)[0]
                DTYPE_OF[_dev] = _m.get("devicetype", "?")
def DTITLE(dev): return f"{DTYPE_OF.get(dev, '?')} {dev}"
def label_devices(df):
    """Display helper: prepend 'devicetype', rename 'device' -> 'deviceid' (a copy)."""
    if "device" not in df.columns: return df
    out = df.copy(); out.insert(0, "devicetype", out["device"].map(lambda d: DTYPE_OF.get(d, "?")))
    return out.rename(columns={"device": "deviceid"})

# SHARED CONFIG — the SAME BAND + QC/extraction params raw_spectra_review saved and
# quality_filtering_dashboard reads, from <ROOT>/electropycal_analysis_config.json (defaults if none).
from electropycal.analysis_config import load_analysis_config, resolve_band, AnalysisConfig
CFG = load_analysis_config(ROOT) if ROOT is not None else AnalysisConfig()
# --- optional inline parameter overrides (uncomment for a one-off change to THIS notebook only) ---
# CFG.band = (10.0, 10_000.0); CFG.peak_method = "chord"; CFG.acceptance = "monotonic"

# featureset FEAT: one row per (device, channel, timepoint, concentration) — QC-gated
if ROOT is None:
    from electropycal.data.synthetic import make_dataset
    _ds = make_dataset(random_state=0)
    FEAT = pd.DataFrame(_ds.X, columns=_ds.feature_names)
    FEAT["channel"] = _ds.channel; FEAT["timepoint"] = _ds.timepoint
    FEAT["concentration"] = _ds.concentration; FEAT["NormIpeak"] = _ds.y
else:
    from electropycal.features.extract import extract_dataset, FEATURE_SCHEMA_VERSION
    BAND = resolve_band(ROOT, CFG)     # shared-config band ("auto" -> recommended_band(ROOT))
    # --- optional feature store: extract ONCE from raw, then reuse on later runs -------------------
    # Standalone is preserved — this only caches THIS notebook's own extract; it never reads another
    # notebook's parquet. The key hashes the extraction params AND the feature-schema version, so it
    # re-extracts automatically when a param or the feature set changes (matches diagnostics_review).
    FORCE_REEXTRACT = False            # True = always rebuild the featureset from raw
    _key = hashlib.md5(json.dumps({"band": [float(BAND[0]), float(BAND[1])], "peak": CFG.peak_method,
        "acc": CFG.acceptance, "mono_tol": CFG.mono_tol, "snr": CFG.min_norm_snr,
        "rmin": CFG.monotonic_r_min, "reps": CFG.max_reps, "d0": True,
        "schema": FEATURE_SCHEMA_VERSION}, sort_keys=True).encode()).hexdigest()[:8]
    CACHE = pathlib.Path(ROOT) / f"discovery_featureset__{_key}.parquet"
    if CACHE.exists() and not FORCE_REEXTRACT:
        FEAT = pd.read_parquet(CACHE)
        print(f"loaded cached featureset: {CACHE.name}  ({len(FEAT)} rows x {FEAT.shape[1]} cols)  |  "
              f"BAND = ({BAND[0]:.0f}, {BAND[1]:.0f}) Hz  peak={CFG.peak_method}  acceptance={CFG.acceptance}  "
              f"(set FORCE_REEXTRACT=True to rebuild from raw)")
    else:
        _t = time.time()
        FEAT = extract_dataset(ROOT, band=BAND, peak_method=CFG.peak_method, acceptance=CFG.acceptance,
                               mono_tol=CFG.mono_tol, min_norm_snr=CFG.min_norm_snr,
                               monotonic_r_min=CFG.monotonic_r_min, max_reps=CFG.max_reps,
                               n_jobs=N_JOBS, progress=True)   # parallel + per-session ETA
        try:
            FEAT.to_parquet(CACHE); _cmsg = f"  ->  cached {CACHE.name}"
        except Exception as e:
            _cmsg = f"  (cache write skipped — install pyarrow to enable: {e})"
        print(f"extract_dataset: {len(FEAT)} rows x {FEAT.shape[1]} cols in {time.time()-_t:.0f}s  |  "
              f"BAND = ({BAND[0]:.0f}, {BAND[1]:.0f}) Hz  peak={CFG.peak_method}  acceptance={CFG.acceptance}{_cmsg}")

OUT = tempfile.mkdtemp()       # per-condition model bundles + metrics are written here
_bstr = f"({BAND[0]:.0f}, {BAND[1]:.0f}) Hz" if ROOT is not None else "n/a (synthetic)"
print(f"ROOT = {ROOT}  |  PROFILE = {'FAST' if PROFILE is FAST else 'full'}  |  TARGET = {TARGET}")
print("discovery parameters (from shared config, else defaults):")
print(f"  BAND = {_bstr}   peak_method = {CFG.peak_method}   acceptance = {CFG.acceptance}")
print(f"  mono_tol(EIS.2) = {CFG.mono_tol}   min_norm_snr(FSCV SNR) = {CFG.min_norm_snr}   "
      f"monotonic_r_min(FSCV.1) = {CFG.monotonic_r_min}   max_reps = {CFG.max_reps}")
print(f"  QC gate feeding discovery: EIS A/B/C + acceptance='{CFG.acceptance}' "
      f"(a channel-timepoint must pass EIS quality AND the acceptance test to enter the featureset)")
print("  features are D0-normalized per sensor (drift from each sensor's earliest timepoint, "
      "); NormIpeak<0 dose rows are dropped (drop_negative) before the featureset")

## 1. Data selection — device type / id / channel / timepoint / concentration
Restrict which rows feed discovery. Each filter defaults to **all available**. `DEVICE_TYPES`
(`neurostring` / `cfme`), `DEVICES` (ids), `CHANNELS`, `TIMEPOINTS` (day integers), and
`CONCENTRATIONS` (nM) are ANDed. The selected rows become the `data` (`RunData`) every batch runs
on — sensor identity is `(device, channel)`, so different devices are kept distinct.

**`TARGET`** (set in Setup) picks *what* is predicted. `"normipeak"` (default) is the per-dose
`NormIpeak`. `"sensitivity"` collapses each sensor-timepoint to one sample whose target is its
**dose-response slope** (`NormIpeak` vs `log10[DA]`) — the quantity that drifts and that
recalibration needs; the same batch queue then predicts that slope from electrode state. RMSEP in
the ranking is on the target's own scale (NormIpeak vs slope), so compare runs *within* a `TARGET`.

In [ ]:
DEVICE_TYPES   = None          # local selection (None = all); or ["neurostring"], ["cfme"]
DEVICES        = None          # None = all device ids; or ["2-2", "3-2"]  (run discovery across many)
CHANNELS       = None          # None = all; or [1, 3, 4]
TIMEPOINTS     = None          # None = all; or [0, 4, 6] (day integers)
CONCENTRATIONS = None          # None = all; or [100.0, 1000.0, 5000.0] (nM)

sel = FEAT
if DEVICE_TYPES is not None and "device" in sel:
    sel = sel[sel.device.map(lambda d: DTYPE_OF.get(d, "?")).isin(DEVICE_TYPES)]
if DEVICES is not None and "device" in sel:       sel = sel[sel.device.isin(DEVICES)]
if CHANNELS is not None:                          sel = sel[sel.channel.isin(CHANNELS)]
if TIMEPOINTS is not None:                        sel = sel[sel.timepoint.isin([float(t) for t in TIMEPOINTS])]
if CONCENTRATIONS is not None:                    sel = sel[sel.concentration.isin([float(c) for c in CONCENTRATIONS])]
sel = sel.reset_index(drop=True)
# TARGET: "normipeak" = one row per (sensor, timepoint, concentration) -> predict per-dose NormIpeak
# (the diagnostic baseline; ill-posed from dose-invariant features). The per-sensor-timepoint
# calibration-curve targets collapse to one row per (sensor, timepoint) and are the deployable,
# dose-resolved signals — the same discovery queue runs on each:
#   "sensitivity" (dose-response slope) | "sensitivity_intercept" | "sensitivity_curvature"
#   "sat_imax"/"sat_kd"/"sat_logkd" (Langmuir)  |  "hill_imax"/"hill_kd"/"hill_n" (Hill)
CURVE_TARGETS = {"sensitivity", "sensitivity_intercept", "sensitivity_curvature",
                 "sat_imax", "sat_kd", "sat_logkd", "hill_imax", "hill_kd", "hill_n",
                 "power_a", "power_beta"}
if TARGET in CURVE_TARGETS:
    from electropycal.features.targets import sensitivity_featureset
    _sens = sensitivity_featureset(sel)                          # fits linear/quadratic/Langmuir/Hill, SNR-weighted
    _sens = _sens[np.isfinite(_sens[TARGET])] if TARGET in _sens else _sens
    data = RunData.from_frame(_sens, target=TARGET)
    print(f"TARGET={TARGET}: {len(sel)} dose rows -> {len(_sens)} sensor-timepoint samples")
else:
    data = RunData.from_frame(sel)

# guard the zero-fold trap: forward-chained CV needs min_train_times prior timepoints
# BEFORE a test point, i.e. >= min_train_times + 1 distinct timepoints. With fewer, the
# queue silently yields 0 folds and nan. effective_min_train_times() is the SAME cap the
# library applies inside run_condition — we apply it here too so the value is right in the
# prints below (the in-library cap then no-ops). Set it visibly on PROFILE.
import dataclasses as _dc
_n_tp = int(sel.timepoint.nunique()) if "timepoint" in sel else 0
_eff = effective_min_train_times(PROFILE.min_train_times, _n_tp)
if _n_tp < 2:
    print(f"WARNING: only {_n_tp} timepoint(s) selected — forward-chained CV needs >= 2; "
          f"discovery will produce no folds. Widen TIMEPOINTS or add more sessions.")
elif _eff != PROFILE.min_train_times:
    print(f"NOTE: {_n_tp} timepoints selected, but PROFILE.min_train_times={PROFILE.min_train_times} "
          f"needs >= {PROFILE.min_train_times + 1} timepoints (would give 0 folds -> nan). "
          f"Capping min_train_times to {_eff} so CV yields folds.")
    PROFILE = _dc.replace(PROFILE, min_train_times=_eff)

_dev = sorted(sel.device.unique()) if "device" in sel else ["(synthetic)"]
print(f"selected {len(sel)} rows  |  devices: " +
      ", ".join(DTITLE(d) for d in _dev) +
      f"  |  channels {sorted(sel.channel.unique())}  |  timepoints {sorted(int(t) for t in sel.timepoint.unique())}"
      f"  |  concentrations {sorted(sel.concentration.unique())}")

## 2. Validated dataset — available QC-valid data
Everything below is already **QC-gated** (`extract_dataset(acceptance="monotonic")` — the exact gate
`quality_filtering_dashboard` previews) and restricted to your section-1 selection. The table (same
layout as `raw_spectra_review`'s available-data table) lists the surviving sensors per
device-timepoint; the summary line is the global size of what discovery will train/test on.

In [ ]:
_grp = [c for c in ("device", "timepoint") if c in sel.columns]
qc_available = (sel.groupby(_grp)
    .agg(n_channels=("channel", "nunique"),
         channels=("channel", lambda s: sorted(int(x) for x in s.unique())),
         n_conc=("concentration", "nunique"), n_rows=("NormIpeak", "size"))
    .reset_index())
qc_available["timepoint"] = qc_available["timepoint"].map(DLAB)

_nsensors = sel.groupby([c for c in ("device", "channel") if c in sel.columns]).ngroups
print(f"validated dataset (post-QC, post-selection): {len(sel)} feature rows, "
      f"{len(data.feature_names)} features")
print(f"  device types: {sorted(set(DTYPE_OF.get(d, '?') for d in sel.device.unique())) if 'device' in sel else ['(synthetic)']}"
      f"  |  sensors (device,channel): {_nsensors}")
print(f"  timepoints: {[DLAB(t) for t in sorted(sel.timepoint.unique())]}  |  "
      f"concentrations (nM): {sorted(sel.concentration.unique())}")
label_devices(qc_available)

### 2.1 Finalized-dataset overview — what discovery will train on
A visual summary of the **post-QC, post-selection** featureset (also prints the run **provenance**).
Two plots, one per cell:
- **Grid** — every sensor (`device:channel`) × timepoint, colored by #concentrations retained:
  **green = all doses, red = 0** (blank = no valid data).
- **Traces** — `NormIpeak` vs timepoint in **true day spacing**, one line per sensor, a panel per
  concentration (y = mean NormIpeak per sensor-timepoint-dose, per-panel autoscaled) — the raw
  dose-resolved trajectories the model sees. (The QC preview of this same view is in
  `quality_filtering_dashboard` §6.1, reviewable without running discovery.)

In [ ]:
from electropycal.analysis_config import print_provenance
print_provenance(ROOT, target=TARGET)              # config + decisions + QC impact (informational)
plot_qc_grid(sel)

In [ ]:
# NormIpeak vs timepoint, one line per sensor; per-panel y clipped at p99 (or pass ymax=...).
plot_normipeak_per_sensor(sel, yclip_pct=99)

In [ ]:
# per-device average (device outliers excluded before averaging; band = +/-1 s.d.)
plot_normipeak_per_device(sel)

## 3. Testing conditions — the batch queue
Discovery evaluates a queue of **testing conditions**, grouped into batches. The table below lists
every available condition — its `batch`, `task`, `architecture`, `k`-LV grid (optimized in the inner
CV), any additional feature `selector`, and the `evaluation` track (CV scheme). **Batch motivation:**

- **1. Baselines** — plain architectures, no feature selection: the RMSEP floor every later batch is judged against.
- **2. Channel-specific** — per-sensor models (`track="channel"`): each `(device, channel)` gets its own trend; the gap vs shared models measures sensor/LIG heterogeneity.
- **3. P > N feature selection** — VIP / SR / MI / ICC / CARS selectors for the high-dimensional feature regime.
- **4. New-channel generalization** — universal track (`loco`: train on other sensors → test a held-out sensor): the hardest transfer.

In [ ]:
queue = baseline_queue()                    # the implemented architectures/selectors
with pd.option_context("display.max_rows", None):
    display(conditions_table(queue))

Run a batch with `run_batch(n)` — it evaluates that batch's conditions with
**`progress=True`** (timestamped per-fold ETA) and prints an overall batch ETA. Pass
`run_batch(n, only=[...])` to run a subset of a batch's conditions (default: all).

In [ ]:
RUN = BatchRunner(queue, data, OUT, PROFILE)
RUN.print_settings()
# convenience aliases so the batch cells below read naturally:
run_batch, advise, exclude, show = RUN.run, RUN.advise, RUN.exclude, RUN.plot

## 4. Batch 1 — baselines (architectures)
The plain-architecture reference queue (linear / log / weighted / orthogonal / nonlinear PLSR, no
feature selection). Establishes the RMSEP floor every later batch is compared against.

**Customize the conditions between batches** — the batch's default list comes from `baseline_queue()`; before you run the cell you can: run a **subset** with `run_batch(N, only=["cond_name", ...])`; **drop** conditions with `exclude("cond_name", ...)`; or **add** one by appending a `Condition(...)` to `queue` (see the `## 3` table for the names/fields). Change the **response** globally with `TARGET` in Setup (`"normipeak"` ↔ `"sensitivity"`).

In [ ]:
show(run_batch(1), 'Batch 1 — baselines')

## 5. Between batches — review & auto-flag
`advise(n)` flags conditions in the next batch that already look dominated (their best-case CI is
worse than the current leader's worst-case CI). Purely advisory — use `exclude(...)` to prune.

In [ ]:
advise(2)

## 6. Batch 2 — channel-specific
Per-sensor models (`track="channel"`): each `(device, channel)` gets its own trend. The RMSEP gap
vs the shared Batch-1 models measures how much sensor/LIG heterogeneity a common model must absorb.

**Customize the conditions between batches** — the batch's default list comes from `baseline_queue()`; before you run the cell you can: run a **subset** with `run_batch(N, only=["cond_name", ...])`; **drop** conditions with `exclude("cond_name", ...)`; or **add** one by appending a `Condition(...)` to `queue` (see the `## 3` table for the names/fields). Change the **response** globally with `TARGET` in Setup (`"normipeak"` ↔ `"sensitivity"`).

In [ ]:
show(run_batch(2), 'Batch 2 — channel-specific')

## 7. Batch 3 — P > N feature selection
Adds feature selectors (VIP / SR / MI / ICC / CARS) for the high-dimensional feature regime.

**Customize the conditions between batches** — the batch's default list comes from `baseline_queue()`; before you run the cell you can: run a **subset** with `run_batch(N, only=["cond_name", ...])`; **drop** conditions with `exclude("cond_name", ...)`; or **add** one by appending a `Condition(...)` to `queue` (see the `## 3` table for the names/fields). Change the **response** globally with `TARGET` in Setup (`"normipeak"` ↔ `"sensitivity"`).

In [ ]:
show(run_batch(3), 'Batch 3 — feature selection')

## 8. Batch 4 — new-channel generalization
Universal track (`loco`: train on other sensors → test a held-out sensor). The hardest transfer —
its RMSEP vs the channel track is the direct measure of cross-sensor/device generalization.

**Customize the conditions between batches** — the batch's default list comes from `baseline_queue()`; before you run the cell you can: run a **subset** with `run_batch(N, only=["cond_name", ...])`; **drop** conditions with `exclude("cond_name", ...)`; or **add** one by appending a `Condition(...)` to `queue` (see the `## 3` table for the names/fields). Change the **response** globally with `TARGET` in Setup (`"normipeak"` ↔ `"sensitivity"`).

In [ ]:
show(run_batch(4), 'Batch 4 — new channels')